# Hugging Face 모델 탐색 실습

이번 실습에서는 Hugging Face에서 제공하는 다양한 인공지능 모델을 직접 찾아보고, Colab에서 실행해보겠습니다.

오늘은 모델을 직접 학습시키는 것이 아니라, 이미 학습되어 공개된 모델을 불러와 사용하는 방법을 연습합니다.

이번 시간에 실습할 task는 다음 4가지입니다.

1. 감정분석
2. 번역
3. 한국어 요약
4. 문장 유사도 비교


#라이브러리 설치

Hugging Face 모델을 사용하기 위해 필요한 라이브러리를 먼저 설치하겠습니다.

아래 코드는 실습에 필요한 기본 라이브러리를 설치하는 코드입니다.
처음 한 번만 실행하면 됩니다.

transformers: Hugging Face 모델을 불러오고 실행하는 라이브러리입니다.
sentencepiece: 일부 번역 및 요약 모델에서 사용하는 토크나이저 라이브러리입니다.
sentence-transformers: 문장을 벡터로 바꾸고 유사도를 계산할 때 사용하는 라이브러리입니다.

In [1]:
!pip install -q transformers sentencepiece sentence-transformers accelerate

In [2]:
import transformers
import sentence_transformers

print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

transformers: 5.12.1
sentence-transformers: 5.6.0


#Hugging Face 모델 검색 방법

Hugging Face에는 다양한 인공지능 모델이 공개되어 있습니다.

이번 실습에서는 원하는 task에 맞는 모델을 직접 검색해보고, Colab에서 실행해보겠습니다.

모델을 찾는 순서는 다음과 같습니다.

1. Hugging Face 사이트에 접속합니다.
2. 상단 메뉴에서 `Models`를 클릭합니다.
3. 왼쪽 메뉴에서 원하는 `Task`를 선택합니다.
4. 검색창에 원하는 키워드를 입력합니다.
5. 마음에 드는 모델 페이지에 들어갑니다.
6. 모델 이름을 복사합니다.
7. Colab 코드의 `model_name` 부분에 붙여넣습니다.

모델 이름은 보통 다음과 같은 형태입니다.

```text
사용자명/모델명
```

예를 들어 다음과 같은 이름을 사용할 수 있습니다.

```text
distilbert-base-uncased-finetuned-sst-2-english
eenzeenee/t5-base-korean-summarization
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
```

처음에는 어떤 모델을 골라야 할지 헷갈릴 수 있습니다.
그럴 때는 다운로드 수가 많거나, 모델 카드에 사용 예시가 잘 적혀 있는 모델을 먼저 선택해보면 좋습니다.


#1. 감정분석
- sentiment-analysis
- text-classificatio

In [3]:
from transformers import pipeline

# Load the classification pipeline with the specified model
pipe = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis")

# Classify a new sentence
sentence = "I love this product! It's amazing and works perfectly."
result = pipe(sentence)

# Print the result
print(result)


config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

[{'label': 'Very Positive', 'score': 0.5586305856704712}]


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "tabularisai/multilingual-sentiment-analysis"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

def predict_sentiment(texts):
    inputs = tokenizer(texts, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    sentiment_map = {0: "Very Negative", 1: "Negative", 2: "Neutral", 3: "Positive", 4: "Very Positive"}
    return [sentiment_map[p] for p in torch.argmax(probabilities, dim=-1).tolist()]

texts = [
    # English
    "I absolutely love the new design of this app!", "The customer service was disappointing.", "The weather is fine, nothing special.",
    # Korean
    "이 가게의 케이크는 정말 맛있어요!", "서비스가 너무 별로였어요.", "날씨가 그저 그렇네요.",
]

for text, sentiment in zip(texts, predict_sentiment(texts)):
    print(f"Text: {text}\nSentiment: {sentiment}\n")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: I absolutely love the new design of this app!
Sentiment: Very Positive

Text: The customer service was disappointing.
Sentiment: Negative

Text: The weather is fine, nothing special.
Sentiment: Neutral

Text: 이 가게의 케이크는 정말 맛있어요!
Sentiment: Positive

Text: 서비스가 너무 별로였어요.
Sentiment: Negative

Text: 날씨가 그저 그렇네요.
Sentiment: Neutral



#2. 번역
- translation

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "trillionlabs/Tri-1.8B-Translation"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

text = "안녕하세요"
messages = [
    {"role": "user", "content": f"Translate the following Korean text into English:\n{text} <en>"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

translation = tokenizer.decode(outputs[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

print(f"Korean: {text}")
print(f"English: {translation}")

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.89M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/4.14k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/291 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/18.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/228 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/145 [00:00<?, ?B/s]

Korean: 안녕하세요
English: Hello


#3. 요약
- summarization
- text2text-generation

In [7]:
import torch
from transformers import PreTrainedTokenizerFast
from transformers import BartForConditionalGeneration

tokenizer = PreTrainedTokenizerFast.from_pretrained('gogamza/kobart-summarization')
model = BartForConditionalGeneration.from_pretrained('gogamza/kobart-summarization')

text = "과거를 떠올려보자. 방송을 보던 우리의 모습을. 독보적인 매체는 TV였다. 온 가족이 둘러앉아 TV를 봤다. 간혹 가족들끼리 뉴스와 드라마, 예능 프로그램을 둘러싸고 리모컨 쟁탈전이 벌어지기도  했다. 각자 선호하는 프로그램을 ‘본방’으로 보기 위한 싸움이었다. TV가 한 대인지 두 대인지 여부도 그래서 중요했다. 지금은 어떤가. ‘안방극장’이라는 말은 옛말이 됐다. TV가 없는 집도 많다. 미디어의 혜 택을 누릴 수 있는 방법은 늘어났다. 각자의 방에서 각자의 휴대폰으로, 노트북으로, 태블릿으로 콘텐츠 를 즐긴다."

raw_input_ids = tokenizer.encode(text)
input_ids = [tokenizer.bos_token_id] + raw_input_ids + [tokenizer.eos_token_id]

summary_ids = model.generate(torch.tensor([input_ids]))
tokenizer.decode(summary_ids.squeeze().tolist(), skip_special_tokens=True)


tokenizer.json:   0%|          | 0.00/682k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/4.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1616: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'TV가 없는 집도 많아지고 미디어의 혜 택을 누릴 수 있는 방법은 늘어났다.'

#4. 문장 유사도 비교
- sentence-similarity
- sentence-transformers 라이브러리 사용

In [8]:
from sentence_transformers import SentenceTransformer, InputExample, losses
import pandas as pd
from sentence_transformers import SentenceTransformer, InputExample
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, util

model_name="Sakil/sentence_similarity_semantic_search"
model = SentenceTransformer(model_name)
sentences = ['A man is eating food.',
          'A man is eating a piece of bread.',
          'The girl is carrying a baby.',
          'A man is riding a horse.',
          'A woman is playing violin.',
          'Two men pushed carts through the woods.',
          'A man is riding a white horse on an enclosed ground.',
          'A monkey is playing drums.',
          'Someone in a gorilla costume is playing a set of drums.'
          ]

#Encode all sentences
embeddings = model.encode(sentences)

#Compute cosine similarity between all pairs
cos_sim = util.cos_sim(embeddings, embeddings)

#Add all pairs to a list with their cosine similarity score
all_sentence_combinations = []

for i in range(len(cos_sim)-1):

    for j in range(i+1, len(cos_sim)):

        all_sentence_combinations.append([cos_sim[i][j], i, j])

#Sort list by the highest cosine similarity score

all_sentence_combinations = sorted(all_sentence_combinations, key=lambda x: x[0], reverse=True)

print("Top-5 most similar pairs:")

for score, i, j in all_sentence_combinations[0:5]:

    print("{} \t {} \t {:.4f}".format(sentences[i], sentences[j], cos_sim[i][j]))


/tmp/ipykernel_1237/2553174365.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.75k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Top-5 most similar pairs:
A man is eating food. 	 A man is eating a piece of bread. 	 0.8160
A monkey is playing drums. 	 Someone in a gorilla costume is playing a set of drums. 	 0.7347
A man is riding a horse. 	 A man is riding a white horse on an enclosed ground. 	 0.7330
A man is riding a horse. 	 Two men pushed carts through the woods. 	 0.4694
A man is eating a piece of bread. 	 A monkey is playing drums. 	 0.4437
